# 02 — Pré-processamento dos Dados

## Objetivo

Este notebook realiza o pré-processamento dos dados para a etapa de modelagem do problema de classificação de risco de crédito.

A partir das decisões fundamentadas na análise exploratória, serão realizadas as seguintes etapas:

- reconstrução da base de modelagem e da variável-alvo (`TARGET`);
- tratamento da codificação especial de `DAYS_EMPLOYED`;
- tratamento dos valores ausentes;
- criação de variáveis mais interpretáveis de idade e tempo de emprego;
- seleção das variáveis preditoras;
- separação dos dados em conjuntos de treino e teste;
- definição do tratamento das variáveis numéricas e categóricas;
- construção de um pipeline de pré-processamento;
- verificação da base resultante para modelagem.

O pré-processamento será ajustado utilizando apenas os dados de treino, reduzindo o risco de *data leakage* e preservando o conjunto de teste para avaliação posterior dos modelos.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

## 1. Carregamento e reconstrução da base de modelagem

Para que este notebook possa ser executado de forma independente, as bases originais serão novamente carregadas e a variável-alvo será reconstruída utilizando o mesmo critério definido na análise exploratória.

Será considerado `TARGET = 1` o cliente que apresentou pelo menos uma ocorrência de atraso igual ou superior a 30 dias (`STATUS` entre `1` e `5`) no histórico observado. Os demais clientes serão classificados como `TARGET = 0`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw")

application = pd.read_csv(DATA_DIR / "application_record.csv")
credit = pd.read_csv(DATA_DIR / "credit_record.csv")

ids_comuns = set(application['ID']).intersection(set(credit['ID']))

credit_comum = credit[credit['ID'].isin(ids_comuns)].copy()

ids_maus_pagadores = set(
    credit_comum.loc[
        credit_comum['STATUS'].isin(['1', '2', '3', '4', '5']),
        'ID'
    ]
)

target = pd.DataFrame({'ID': list(ids_comuns)})
target['TARGET'] = target['ID'].isin(ids_maus_pagadores).astype(int)

df = application.merge(target, on='ID', how='inner')

print(f'Dimensão da base: {df.shape}')
print('\nDistribuição do TARGET:')
print(df['TARGET'].value_counts())

## 2. Tratamento inicial das variáveis

A análise exploratória identificou duas características que exigem tratamento antes da modelagem.

`DAYS_BIRTH` representa a idade em dias negativos. Essa variável será convertida para idade em anos (`AGE_YEARS`), facilitando sua interpretação.

Em `DAYS_EMPLOYED`, o valor `365243` foi identificado como uma codificação especial associada principalmente aos pensionistas e não representa literalmente o tempo de emprego. Esses registros serão convertidos para valores ausentes antes da criação de `YEARS_EMPLOYED`.

Os valores ausentes de `OCCUPATION_TYPE` serão representados pela categoria `Unknown`. Essa decisão preserva os registros e permite que a ausência de informação ocupacional seja considerada explicitamente pelo modelo.

In [ ]:
df_clean = df.copy()

# Idade em anos
df_clean['AGE_YEARS'] = (
    -df_clean['DAYS_BIRTH'] / 365.25
)

# Substitui a codificação especial por NaN e converte para anos
df_clean['YEARS_EMPLOYED'] = np.where(
    df_clean['DAYS_EMPLOYED'] == 365243,
    np.nan,
    -df_clean['DAYS_EMPLOYED'] / 365.25
)

# Ausência de ocupação passa a ser uma categoria explícita
df_clean['OCCUPATION_TYPE'] = (
    df_clean['OCCUPATION_TYPE'].fillna('Unknown')
)

# As variáveis originais em dias não serão utilizadas
df_clean = df_clean.drop(
    columns=['DAYS_BIRTH', 'DAYS_EMPLOYED']
)

print(f'Dimensão após o tratamento: {df_clean.shape}')

print('\nValores ausentes:')
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

print('\nResumo das novas variáveis:')
display(
    df_clean[['AGE_YEARS', 'YEARS_EMPLOYED']]
    .describe()
    .round(2)
)

## 3. Seleção das variáveis preditoras

Antes da divisão entre treino e teste, serão definidas as variáveis que participarão da modelagem.

A coluna `ID` será excluída por funcionar apenas como identificador do cliente, sem significado preditivo direto.

A variável `FLAG_MOBIL` também será removida, pois a análise exploratória mostrou que possui valor constante para todos os registros e, portanto, não fornece informação capaz de discriminar as classes.

As variáveis auxiliares criadas exclusivamente para a análise exploratória não fazem parte desta base. As demais características cadastrais serão inicialmente preservadas para a etapa de modelagem.

In [ ]:
X = df_clean.drop(
    columns=['ID', 'FLAG_MOBIL', 'TARGET']
)

y = df_clean['TARGET']

print(f'Dimensão de X: {X.shape}')
print(f'Dimensão de y: {y.shape}')

print('\nVariáveis preditoras:')
for coluna in X.columns:
    print(f'- {coluna}')

## 4. Divisão entre treino e teste

A base será dividida em conjuntos de treino e teste antes do ajuste das etapas de imputação, padronização e codificação das variáveis categóricas.

Serão destinados **80% dos registros para treino** e **20% para teste**. A divisão será estratificada pela variável `TARGET`, preservando aproximadamente a mesma proporção de bons e maus pagadores nos dois conjuntos.

O conjunto de teste permanecerá separado durante o desenvolvimento dos modelos e será utilizado posteriormente para a avaliação final.

Essa ordem é importante para evitar *data leakage*: parâmetros utilizados no pré-processamento, como medianas e médias, devem ser aprendidos exclusivamente a partir dos dados de treino.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test:  {y_test.shape}')

print('\nDistribuição percentual do TARGET:')
print(f'Treino:')
print((y_train.value_counts(normalize=True) * 100).round(2))

print('\nTeste:')
print((y_test.value_counts(normalize=True) * 100).round(2))

## 5. Definição das etapas de pré-processamento

As variáveis preditoras exigem tratamentos diferentes conforme sua natureza.

As variáveis numéricas contínuas ou de contagem serão submetidas à imputação de valores ausentes pela mediana e à padronização.
A padronização coloca as variáveis numéricas em escalas comparáveis, evitando que atributos com valores de maior magnitude tenham influência desproporcional em algoritmos sensíveis à escala, como a Regressão Logística. A transformação será ajustada exclusivamente sobre os dados de treino para evitar data leakage.

As variáveis categóricas serão transformadas por meio de *One-Hot Encoding*, permitindo sua utilização pelos algoritmos de Machine Learning sem impor uma relação ordinal artificial entre as categorias.

As variáveis binárias já representadas numericamente por `0` e `1` serão mantidas sem transformação.

In [ ]:
numeric_features = [
    'CNT_CHILDREN',
    'AMT_INCOME_TOTAL',
    'CNT_FAM_MEMBERS',
    'AGE_YEARS',
    'YEARS_EMPLOYED'
]

binary_features = [
    'FLAG_WORK_PHONE',
    'FLAG_PHONE',
    'FLAG_EMAIL'
]

categorical_features = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE'
]

print(f'Numéricas: {len(numeric_features)}')
print(f'Binárias: {len(binary_features)}')
print(f'Categóricas: {len(categorical_features)}')

total_features = (
    len(numeric_features)
    + len(binary_features)
    + len(categorical_features)
)

print(f'\nTotal: {total_features}')
print(f'Variáveis em X: {X.shape[1]}')

### Pipeline de transformação

O pré-processamento será encapsulado em um `ColumnTransformer`, permitindo aplicar transformações específicas a cada grupo de variáveis.

Para as variáveis numéricas, os valores ausentes serão imputados pela mediana calculada exclusivamente no conjunto de treino. Também será criado um indicador da ausência original e, em seguida, as variáveis serão padronizadas.

Para as variáveis categóricas será utilizado *One-Hot Encoding*, configurado para ignorar categorias eventualmente presentes no conjunto de teste que não tenham sido observadas durante o treinamento.

As variáveis binárias serão mantidas em sua representação original.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='median',
                add_indicator=True
            )
        ),
        (
            'scaler',
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore'
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_transformer,
            numeric_features
        ),
        (
            'cat',
            categorical_transformer,
            categorical_features
        ),
        (
            'bin',
            'passthrough',
            binary_features
        )
    ]
)

preprocessor

## 6. Validação do pré-processamento

Antes da etapa de modelagem, o pré-processador será ajustado exclusivamente sobre o conjunto de treino para verificar o funcionamento das transformações definidas.

Essa execução possui finalidade de validação. Na etapa de modelagem, o pré-processamento será incorporado aos pipelines dos algoritmos, permitindo que as transformações sejam ajustadas corretamente dentro de cada processo de treinamento e validação cruzada.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

print(f'Formato original de X_train: {X_train.shape}')
print(f'Formato após pré-processamento: {X_train_processed.shape}')

print(
    f'\nValores ausentes em YEARS_EMPLOYED antes do tratamento: '
    f'{X_train["YEARS_EMPLOYED"].isna().sum():,}'
)

# Verifica valores ausentes após o pré-processamento
if hasattr(X_train_processed, 'toarray'):
    valores_processados = X_train_processed.toarray()
else:
    valores_processados = X_train_processed

print(
    f'Valores ausentes após o pré-processamento: '
    f'{np.isnan(valores_processados).sum():,}'
)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print(f'Total de atributos após o pré-processamento: {len(feature_names)}')

print('\nPrimeiros 20 atributos:')
for nome in feature_names[:20]:
    print(f'- {nome}')

## 7. Síntese do pré-processamento

O pré-processamento preparou a base para a etapa de modelagem preservando as decisões fundamentadas durante a análise exploratória.

Foram realizadas as seguintes definições:

- conversão de `DAYS_BIRTH` para `AGE_YEARS`;
- tratamento da codificação especial de `DAYS_EMPLOYED` e criação de `YEARS_EMPLOYED`;
- representação dos valores ausentes de `OCCUPATION_TYPE` pela categoria `Unknown`;
- exclusão de `ID`, por seu caráter identificador, e de `FLAG_MOBIL`, por ser constante;
- separação estratificada da base em 80% para treino e 20% para teste;
- imputação das variáveis numéricas pela mediana, com criação de indicador de ausência;
- padronização das variáveis numéricas;
- aplicação de *One-Hot Encoding* às variáveis categóricas;
- preservação das variáveis binárias originalmente representadas por 0 e 1.

Após as transformações, as 16 variáveis preditoras originais resultaram em **55 atributos**, sem valores ausentes na matriz processada.

O pré-processador foi ajustado sobre o conjunto de treino apenas para validar as transformações. Na etapa de modelagem, essas operações serão incorporadas diretamente aos pipelines dos algoritmos, permitindo que sejam ajustadas exclusivamente sobre os dados disponíveis em cada treinamento e validação cruzada, reduzindo o risco de *data leakage*.

O conjunto de teste permanecerá reservado para a avaliação final dos modelos.